# Low-Correlation Portfolio Analytics

Runs the portfolio analysis on the 8 lowest-correlated assets out of the 36 analysed so far with at least 300 days of Hyperliquid history. Each round removed the asset with the highest average absolute correlation to those remaining, using 317 common daily returns to 2026-11-04. Outputs go to `data_low_corr/`, separate from the wallet-based analysis.


## Load implementation

`LOW_CORRELATION_ASSETS` replaces wallet discovery; re-check the selection periodically because correlations change.


In [ ]:
import sys
sys.path.insert(0, '..')

from portfolio_analytics_implementation import PortfolioAnalytics

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]

LOW_CORRELATION_ASSETS = [
    'FOGO', 'HEMI', 'INIT', 'TRX', 'VVV', 'ZRO', 'xyz:AAPL', 'xyz:GOOGL',
]

analysis=PortfolioAnalytics(
    data_dir='data_low_corr',
    wallet_addresses=WALLET_ADDRESSES,
    assets=LOW_CORRELATION_ASSETS,
)


## Market data refresh

Uses Hyperliquid daily candles for prices and CoinGecko for market-cap and volume data where available.


In [ ]:
analysis.refresh_market_data()


## Price history

Loads the selected assets plus BTC for benchmark comparisons.


In [ ]:
analysis.load_price_history()


## Daily returns and correlations

Calculates the complete daily-return sample and Pearson correlations.


In [ ]:
analysis.calculate_returns()


## Correlation heatmaps

Displays separate daily return correlation heatmaps for Crypto (main-exchange and spot), Tradfi (HIP-3 stocks, indices, commodities, FX, pre-IPO, and rates), and HIP-3 (any other HIP-3 market). Each asset appears in one group, and each group uses all the dates its assets share.


In [ ]:
analysis.plot_correlations()


## Constrained long/short full-Kelly portfolio

Optimizes signed asset weights from historical returns; the remaining net balance is held as cash. Long/short signs are chosen by the optimizer and are not forced to match current wallet position sides.


In [ ]:
analysis.optimize_portfolio()


## Portfolio performance and risk metrics

Displays the Plotly portfolio-vs-BTC chart and performance and risk tables. Return and volatility annualization use 365 calendar days. Arithmetic annualized return is the daily mean multiplied by 365, not a forecast or compounded return. Historical VaR is an empirical quantile; Bootstrap VaR resamples the observed daily returns 10,000 times and reports the median with a 90% interval, so it cannot exceed the largest loss in the sample. The observation and tail counts show how limited the 99% historical estimate is.


In [ ]:
analysis.analyze_performance()


## Factor model

Fits exposures to BTC, BTC momentum (long or short BTC by the sign of its lagged 30-day return), BTC realized volatility (lagged 30-day), and a funding spread (long low-funding, short high-funding held perps). CoinGecko size, momentum, and turnover factors are added when at least 60 complete days are available.


In [ ]:
analysis.fit_factor_model()


## Alpha split-sample test

Splits the factor-model dates in half. Out-of-sample rows fit the Kelly weights on one half and measure factor-adjusted alpha on the other, so the weights never see the dates they are judged on.


In [ ]:
analysis.test_alpha_out_of_sample()


## Save analysis outputs

Writes calculated tables and series to CSV files in `data/`.


In [ ]:
analysis.save_outputs()
